# D'QUILANE — Tester Notebook

Notebook ini khusus buat **load model yang sudah ditraining** & generate text.
Gak ada training di sini, jadi cepat banget (~10 detik).

## Cara pakai:
1. Training dulu pakai `dquilane_balita.ipynb` (Run All)
2. Setelah training selesai, output `dquilane_balita.pt` tersimpan otomatis
3. Notebook ini akan load model itu & test generate

In [ ]:
# ===== CELL 1: SETUP =====
import subprocess, sys
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', '-q'])
    import torch
import torch.nn as nn
import torch.nn.functional as F
import math, os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')

In [ ]:
# ===== CELL 2: LOAD MODEL =====
# Model harus sudah ada di output/dquilane_balita.pt
# Kalau belum ada, run dulu dquilane_balita.ipynb

MODEL_PATH = 'output/dquilane_balita.pt'
# Alternatif kalau mau pakai model dari run sebelumnya di Kaggle:
# MODEL_PATH = '../input/<dataset-name>/dquilane_balita.pt'

if not os.path.exists(MODEL_PATH):
    print(f'[ERROR] Model gak ketemu di {MODEL_PATH}')
    print('Cara dapatkan:')
    print('  1. Run dulu dquilane_balita.ipynb (training notebook)')
    print('  2. Atau upload file .pt via Add Input di kanan')
    raise FileNotFoundError(MODEL_PATH)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
CONFIG = ckpt['config']
stoi  = ckpt['stoi']
itos  = {int(k): v for k, v in ckpt['itos'].items()}
VOCAB = CONFIG['vocab_size']

print(f'\n=== Model loaded ===')
print(f'Config: {CONFIG}')
print(f'Vocab: {VOCAB} karakter')
print(f'Sample tokens: I={stoi.get("I", "?")}, n={stoi.get("n", "?")}, d={stoi.get("d", "?")}')

def encode(s):
    return [stoi[c] for c in s if c in stoi]

def decode(ids):
    return ''.join(itos.get(int(i), '?') for i in ids)

In [ ]:
# ===== CELL 3: DEFINE MODEL CLASS =====
# Harus sama dengan yang di training notebook

class Attention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert d % h == 0
        self.h = h
        self.dh = d // h
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.h, self.dh)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out(out)

class FFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.val  = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.val(x))

class Block(nn.Module):
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.norm1 = nn.RMSNorm(d)
        self.attn  = Attention(d, h)
        self.norm2 = nn.RMSNorm(d)
        self.ffn   = FFN(d, d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class DQuilane(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['d_model'])
        self.pos_emb = nn.Embedding(cfg['max_seq'], cfg['d_model'])
        self.block = Block(cfg['d_model'], cfg['n_heads'], cfg['d_ff'])
        self.norm_f = nn.RMSNorm(cfg['d_model'])
        self.head = nn.Linear(cfg['d_model'], cfg['vocab_size'], bias=False)
        self.head.weight = self.tok_emb.weight
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for _ in range(self.cfg['n_layers']):
            x = self.block(x)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg['vocab_size']),
                                    targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new=50, temp=0.8, top_k=10):
        self.eval()
        for _ in range(max_new):
            x = idx if idx.size(1) <= self.cfg['max_seq'] else idx[:, -self.cfg['max_seq']:]
            logits, _ = self(x)
            logits = logits[:, -1, :] / temp
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

# buat instance & load bobot
model = DQuilane(CONFIG).to(DEVICE)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print('Model weights loaded! Ready to generate.')

In [ ]:
# ===== CELL 4: TEST GENERATE (PROMPTS FIXED) =====
# Edit list prompts di bawah utk test berbagai input

PROMPTS = [
    'Indonesia',
    'Belajar',
    'Padi',
    'Teknologi',
    'Mari bangun',
    'Kerja keras',
]

for p in PROMPTS:
    ids = torch.tensor([encode(p)], dtype=torch.long, device=DEVICE)
    out = model.generate(ids, max_new=100, temp=0.7, top_k=10)
    text = decode(out[0].tolist())
    print(f'\n[{p}]')
    print(f'  → {text}')
    print('-' * 60)

In [ ]:
# ===== CELL 5: TEST INTERAKTIF (input manual) =====
# Test dengan prompt sendiri. Run cell ini berulang dengan prompt berbeda.

def generate_text(prompt, max_new=100, temp=0.7, top_k=10):
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    if ids.size(1) == 0:
        return '[ERROR] prompt gak ada karakter yang dikenal'
    out = model.generate(ids, max_new=max_new, temp=temp, top_k=top_k)
    return decode(out[0].tolist())

# Contoh: ganti prompt di sini lalu run
MY_PROMPT = 'Indonesia adalah negara'
print(f'PROMPT: {MY_PROMPT!r}')
print(f'OUTPUT: {generate_text(MY_PROMPT)}')

print()
MY_PROMPT = 'Belajar setiap hari'
print(f'PROMPT: {MY_PROMPT!r}')
print(f'OUTPUT: {generate_text(MY_PROMPT)}')

In [ ]:
# ===== CELL 6: EXPERIMENT — variasi temperature & top_k =====
# Lihat efek parameter sampling ke output

prompt = 'Indonesia'
print(f'PROMPT: {prompt!r}\n')

for temp in [0.3, 0.7, 1.0, 1.5]:
    print(f'--- temp={temp} (makin gede = makin random) ---')
    print(generate_text(prompt, temp=temp, top_k=10))
    print()

for top_k in [3, 10, 30, None]:
    k = top_k if top_k else 999
    print(f'--- top_k={top_k} (makin gede = makin banyak pilihan) ---')
    print(generate_text(prompt, temp=0.7, top_k=k))
    print()

## Tips Pengujian

| Gejala Output | Penyebab | Solusi |
|---------------|----------|--------|
| Sama persis tiap run | temp terlalu rendah | Naikkan temp ke 0.8-1.0 |
| Random parah | temp terlalu tinggi / loss training masih tinggi | Turunkan temp ke 0.3-0.5, train lagi sampai loss <1.0 |
| Karakter aneh / pengulangan | top_k terlalu kecil / model overfit | Naikkan top_k ke 30, atau kurangi epochs |
| Output bagus tapi pendek | max_new kekecilan | Naikkan max_new di cell 4/5 |

## Parameter sampling:
- **temp** (temperature): 0.3 = konservatif, 1.0 = balanced, 1.5+ = liar
- **top_k**: k = 3 = sangat fokus, k = 30 = banyak variasi, None = semua vocab
- **max_new**: berapa token baru yang di-generate